### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 0.2_profile_selection
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es) & Bàrbara Barceló-Llull 
### Data from BioSWOT-Med experiment
# 
**DESCRIPTION**:
 This script performs the first quality control and profile selection on the 
 raw NetCDF files generated in step 0.1. It filters profiles based on specific 
 criteria such as minimum maximum depth, maximum starting depth, minimum number 
 of data points, and maximum allowed fraction of NaNs. It generates CSV logs 
 for both selected and rejected profiles, detailing the reasons for rejection.
#
 INPUT: Directory containing raw *_downcast.nc files.
#
 OUTPUT: CSV files listing selected and rejected profiles, and optionally 
 a folder with the physically copied selected NetCDF files.
### ==============================================================================

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import shutil

# ==========================
# USER SETTINGS (BIOSWOT)
# ==========================
# Input directory containing raw NetCDF files (output from step 0.1)
dir_nc = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\Data\raw") 

# Output directory for the selection results
dir_output = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\Data\selected_profiles")

# Search pattern: Important to use *_downcast.nc if the previous split script was used
# If your files are just named .nc, change this to "*.nc"
pattern = "*_downcast.nc" 

# Selection Criteria (Adjustable)
MIN_ZMAX = 80.0      # Minimum required maximum depth
MAX_ZMIN = 10.0      # Maximum starting depth (profile must start before 10m)
MIN_NPTS = 30        # Minimum number of data points required
MAX_NAN_FRAC = 0.2   # Maximum allowed fraction of NaNs per variable

# Output files
dir_output.mkdir(parents=True, exist_ok=True)
out_csv = dir_output / "selected_profiles.csv"
out_rej = dir_output / "rejected_profiles.csv"

# Option to physically copy the selected files to another folder
copy_selected = False
dir_sel = dir_output / "selected_nc"
if copy_selected:
    dir_sel.mkdir(exist_ok=True)

# ==========================
# PROCESS
# ==========================
rows = []
rej_rows = []

# Verify if the input directory exists
if not dir_nc.exists():
    print(f"❌ ERROR: Cannot find input directory: {dir_nc}")
    print("   Make sure Step 0.1 saved the files there.")
else:
    files = sorted(dir_nc.glob(pattern))
    print(f"Found {len(files)} netCDF files in {dir_nc}")

    count = 0
    for f in files:
        try:
            ds = xr.open_dataset(f)

            # --- READ VARIABLES ---
            # Pressure
            if "pressure" in ds: depth = ds["pressure"].values
            elif "Depth" in ds: depth = ds["Depth"].values
            elif "Press" in ds: depth = ds["Press"].values
            else: raise ValueError("Variable pressure/Depth not found")

            # Temperature
            if "t1" in ds: temp = ds["t1"].values
            elif "Temp" in ds: temp = ds["Temp"].values
            else: temp = np.full_like(depth, np.nan)

            # Conductivity
            if "c1" in ds: cond = ds["c1"].values
            elif "Cond" in ds: cond = ds["Cond"].values
            else: cond = np.full_like(depth, np.nan)

            # Salinity (Solo para control de calidad)
            if "s_raw" in ds: sal = ds["s_raw"].values
            elif "Sal" in ds: sal = ds["Sal"].values
            else: sal = np.full_like(depth, np.nan)

            # --- STATISTICAL CALCULATIONS ---
            if len(depth) > 0:
                zmax = np.nanmax(depth)
                zmin = np.nanmin(depth)
                npts = np.isfinite(depth).sum()
            else:
                zmax, zmin, npts = 0.0, 0.0, 0

            nanT = np.mean(~np.isfinite(temp)) if len(temp) > 0 else 1.0
            nanC = np.mean(~np.isfinite(cond)) if len(cond) > 0 else 1.0
            nanS = np.mean(~np.isfinite(sal)) if len(sal) > 0 else 1.0

            # --- REJECTION CRITERIA CHECK ---
            reasons = []
            if not (zmax >= MIN_ZMAX): reasons.append(f"zmax({zmax:.1f})<{MIN_ZMAX}")
            if not (zmin <= MAX_ZMIN): reasons.append(f"zmin({zmin:.1f})>{MAX_ZMIN}")
            if not (npts >= MIN_NPTS): reasons.append(f"npts({npts})<{MIN_NPTS}")
            if nanT > MAX_NAN_FRAC: reasons.append(f"nanT({nanT:.2f})>{MAX_NAN_FRAC}")
            if nanC > MAX_NAN_FRAC: reasons.append(f"nanC({nanC:.2f})>{MAX_NAN_FRAC}")
            
            row = dict(
                file=f.name,
                stem=f.stem,
                zmin=float(zmin),
                zmax=float(zmax),
                npts=int(npts),
                nanT=float(nanT),
                nanC=float(nanC),
                nanS=float(nanS),
            )

            if len(reasons) == 0:
                rows.append(row)
                if copy_selected:
                    shutil.copy2(f, dir_sel / f.name)
            else:
                row["reasons"] = ";".join(reasons)
                rej_rows.append(row)

            ds.close()
            
            count += 1
            if count % 100 == 0: print(f" ... {count} checked")

        except Exception as e:
            rej_rows.append(dict(file=f.name, stem=f.stem, reasons=f"read_error: {str(e)}"))

    # --- SAVE RESULTS ---
    if len(rows) > 0:
        df_sel = pd.DataFrame(rows).sort_values(["zmax"], ascending=False)
        df_sel.to_csv(out_csv, index=False)
        print(f"\n✅ Selected profiles: {len(df_sel)}")
        print(f"   CSV saved to: {out_csv}")
    else:
        print("\n⚠️ WARNING: No profiles selected! (rows list is empty)")

    if len(rej_rows) > 0:
        df_rej = pd.DataFrame(rej_rows)
        df_rej.to_csv(out_rej, index=False)
        print(f"❌ Rejected profiles: {len(df_rej)}")
        print(f"   CSV saved to: {out_rej}")
    else:
        # Create empty CSV to avoid errors if no profiles were rejected
        pd.DataFrame(columns=['file', 'reasons']).to_csv(out_rej, index=False)

Found 640 netCDF files in C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\Data\raw
 ... 100 checked
 ... 200 checked
 ... 300 checked
 ... 400 checked
 ... 500 checked
 ... 600 checked

✅ Selected profiles: 633
   CSV saved to: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\Data\selected_profiles\selected_profiles.csv
❌ Rejected profiles: 7
   CSV saved to: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing\Data\selected_profiles\rejected_profiles.csv
